# SpaceJEPA v5 · What does the rest of the GPU work cost?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rk0604/ReEntryAI/blob/space-jepa5/StochasticEntrySim/ReEntryAI_JEPA_v5_gpu_cost.ipynb)

A short measurement, not part of the study: it **reads** `MyDrive/ReEntryAI_jepa_v5/` (to see what is already done)
and writes nothing there. It times the real study code on this runtime with the study's frozen settings: E3 model
training, MPPI flights of every model kind (physics `mppi_filtered`, `v2single`, the three E3 arms, `r0_nojepa`), and
the open-loop evaluation. Then it projects the hours and Colab units for all the GPU work the study still has to do,
except RL (the RL cost notebook covers that).

**How to run.** It opens on a **T4 GPU** runtime (keep it: T4 is the study's GPU). **Runtime → Run all.** About
45–90 minutes (≈ 1.5–3 units). The result is printed at the end and saved to `MyDrive/ReEntryAI_v5_gpu_cost/`.
**The notebook disconnects itself** when it finishes or a cell fails, so it never idles on paid units; set
`AUTO_DISCONNECT = False` in the Setup cell to keep the runtime.

The measured models are trained for only a few thousand steps, so they fly worse than the study's models, which
changes flight lengths somewhat: treat the result as ±20 %.

## 0. Settings

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
MODE = os.environ.get("JEPA_V5_MODE", "full")        # "smoke": tiny sizes (the smoke test; numbers meaningless)
TRAIN_STEPS = 2000                                   # E3 training steps timed per arm (a full model is 36,000)

## 1. Setup

Clone the `space-jepa5` code, mount Drive, a scratch folder on local disk, and the auto-disconnect hook. A minute.

In [ ]:
BRANCH = "space-jepa5"
if IN_COLAB:
    REPO = "/content/ReEntryAI_v5"
    if not os.path.exists(REPO):
        !git clone -q -b {BRANCH} https://github.com/rk0604/ReEntryAI.git {REPO}
    else:
        !git -C {REPO} pull -q origin {BRANCH}
    %pip install -q pymsis
    os.chdir(f"{REPO}/StochasticEntrySim")
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
    OUT = f"{DRIVE}/ReEntryAI_v5_gpu_cost"
    V5_ROOT, V3_ROOT = f"{DRIVE}/ReEntryAI_jepa_v5", f"{DRIVE}/ReEntryAI_jepa_v3"
    DATA_CANDIDATES = [f"{DRIVE}/ReEntryAI_jepa_data/wide_6000/episodes_wide_6000.pkl",
                       f"{DRIVE}/ReEntryAI_jepa_data/wide_6000"]
    LOCAL_DATA = "/content/data"
    SCRATCH = "/content/gpu_cost_scratch"
else:
    OUT = os.environ.get("JEPA_V5_GPUCOST_ROOT", "jepa_runs/v5_gpu_cost")
    V5_ROOT = os.environ.get("JEPA_V5_ROOT", "")
    V3_ROOT = os.environ.get("JEPA_V3_ROOT", f"jepa_runs/v3_{MODE}")
    DATA_CANDIDATES = [os.environ.get("JEPA_WIDE_DATA", ""), "jepa_runs/episodes_wide_6000.pkl", "jepa_runs/wide_6000"]
    LOCAL_DATA = os.path.join(OUT, "data")
    SCRATCH = os.path.join(OUT, "scratch")

import json, time
from pathlib import Path
sys.path.insert(0, os.getcwd())
import torch
from jepa import study_v4 as S4, study_v5 as S5, train_v2 as T2, gpucost_v5 as GC

CFG = S5.StudyConfigV5.smoke() if MODE == "smoke" else S5.StudyConfigV5()
if MODE == "smoke":
    TRAIN_STEPS = int(CFG.e3_train["steps"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
CPUS = os.cpu_count() or 2
WORKERS = GC.gpu_workers(DEVICE)                      # the study notebook's GPU worker count
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
torch.set_num_threads(1 if DEVICE == "cuda" else max(1, CPUS // 2))
L3 = S4.v3_layout(V3_ROOT)                            # read-only v3 folder (v2single's models)
DATA_DIRS = [c if os.path.isdir(c) else os.path.dirname(c) for c in DATA_CANDIDATES if c and os.path.exists(c)]
RO = [V3_ROOT] + DATA_DIRS + ([V5_ROOT] if V5_ROOT else [])
L = S5.LayoutV5(SCRATCH, read_only=RO).make()         # scratch only: never the study folder
os.makedirs(OUT, exist_ok=True)
print(f"MODE={MODE} device={DEVICE} ({GPU}) cpus={CPUS} GPU workers={WORKERS}; git {S5._GIT.get()[:10]}; "
      f"result -> {OUT}")
if MODE == "full" and "T4" not in GPU:
    print(f"NOTE: this runtime's GPU is {GPU!r}, not a T4: the projection applies to this GPU, not to a T4.")

# ---- Auto-disconnect: stop paying for the runtime once the work is done or a cell fails ----
AUTO_DISCONNECT = IN_COLAB      # set to False to keep the runtime (e.g. to explore results by hand)

def release_runtime(reason):
    # Flush Drive, then release the Colab runtime so it stops using compute units.
    if not (IN_COLAB and AUTO_DISCONNECT):
        print(f"[auto-disconnect off] {reason}")
        return
    print(f"[auto-disconnect] {reason}. Flushing Drive and releasing the runtime; outputs stay visible.", flush=True)
    try:
        drive.flush_and_unmount()
    except Exception as e:
        print("  Drive flush:", e)
    from google.colab import runtime
    runtime.unassign()

def _release_on_error(result):
    err = result.error_in_exec or result.error_before_exec
    if err is not None and not isinstance(err, KeyboardInterrupt):     # pressing stop is not a failure
        release_runtime(f"a cell failed ({type(err).__name__}: {str(err)[:200]})")

_ip = get_ipython()
for _cb in list(_ip.events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_release_on_error":
        _ip.events.unregister("post_run_cell", _cb)
_ip.events.register("post_run_cell", _release_on_error)
print("auto-disconnect on: the runtime is released when the last cell finishes or a cell fails" if AUTO_DISCONNECT
      else "auto-disconnect off (not on Colab, or switched off): the runtime stays up until you disconnect it")

## 2. Data

The 6,000 training flights, copied to local disk and loaded (every GPU session of the study does this once). A few minutes.

In [ ]:
t0 = time.time()
DATA_PATH = T2.local_copy(T2.find_wide_dataset(DATA_CANDIDATES, download_dir=LOCAL_DATA), LOCAL_DATA)
DATA = S5.load_data_v5(CFG, DATA_PATH, DEVICE)
M = {"hardware": GPU, "workers": WORKERS, "cpus": CPUS, "data_load_s": round(time.time() - t0, 1),
     "train_steps": TRAIN_STEPS}
print(f"{DATA['n_episodes']} flights loaded in {M['data_load_s']:.0f} s")

## 3. E3 training

Each of the four E3 arms trains 2,000 of its 36,000 steps with the study's settings. 10–20 minutes.

In [ ]:
t0 = time.time()
M["training"] = GC.measure_training(CFG, L, DATA, DEVICE, TRAIN_STEPS)
print(f"training measured in {(time.time() - t0) / 60:.1f} min")

## 4. MPPI flights and open loop

One lockstep batch per model kind on the study's GPU workers: `mppi_filtered` and `v2single` on A256 and ORB-1 conditions, the three E3 arms and `r0_nojepa` on A256 (the short-trained models), then one open-loop evaluation per E3 arm. 30–60 minutes.

In [ ]:
SMOKE = {} if MODE == "full" else dict(n_model=2, n_physics=2)
t0 = time.time()
M["mppi"] = GC.measure_mppi(CFG, L, L3, DEVICE, WORKERS, **SMOKE)
M["open_loop"] = GC.measure_open_loop(CFG, L, DATA, DEVICE)
print(f"flights and open loop measured in {(time.time() - t0) / 60:.1f} min")

## 5. Projection

Reads what `MyDrive/ReEntryAI_jepa_v5/` still has to do (read only) and projects the hours and Colab units of all its remaining GPU work except RL, plus its CPU-pool flights at the study's own measured rates. Saved to `MyDrive/ReEntryAI_v5_gpu_cost/`.

In [ ]:
LEFT = GC.study_left(CFG, V5_ROOT, L3, L)
P = GC.project(CFG, M, LEFT)
TEXT = GC.summary_text(M, LEFT, P)
print(TEXT)
stamp = time.strftime("%Y%m%d_%H%M")
name = f"gpu_cost_{GPU.replace(' ', '_')}_{stamp}"
Path(OUT, name + ".json").write_text(json.dumps({"measured": M, "left": LEFT, "projection": P,
                                                 "git": S5._GIT.get()}, indent=1, default=str))
Path(OUT, name + ".txt").write_text(TEXT)
print(f"saved {OUT}/{name}.json and .txt")

## Done: release the runtime

Everything this notebook made is already on Drive. This cell flushes Drive and releases the Colab
runtime, so it stops using compute units (a failed cell does the same). To keep the runtime, set
`AUTO_DISCONNECT = False` in the setup cell before running.

In [ ]:
release_runtime("all cells finished")